# Ejercicio SQL: Cervezas
---
En este notebook realizaremos la creación de la base de datos y la resolución de las consultas utilizando `sqlite3` y `pandas` para visualizar los resultados.

In [1]:
import sqlite3
import pandas as pd

# Conectamos a la base de datos (se creará el archivo si no existe)
conexion = sqlite3.connect('cervezas.db')
cursor = conexion.cursor()

### 1. Creación de Tablas e Inserción de Datos
Preparamos el entorno con los datos proporcionados en el enunciado.

In [2]:
# Borrar tablas si existen para evitar errores al re-ejecutar
cursor.executescript('''
DROP TABLE IF EXISTS REPARTO;
DROP TABLE IF EXISTS CERVEZAS;
DROP TABLE IF EXISTS BARES;
DROP TABLE IF EXISTS EMPLEADOS;

CREATE TABLE CERVEZAS (CodC TEXT PRIMARY KEY, Envase TEXT, Capacidad REAL, Stock INTEGER);
CREATE TABLE BARES (CodB TEXT PRIMARY KEY, Cif TEXT, Nombre TEXT, Localidad TEXT);
CREATE TABLE EMPLEADOS (CodE INTEGER PRIMARY KEY, Nombre TEXT, Sueldo REAL);
CREATE TABLE REPARTO (CodE INTEGER, CodB TEXT, CodC TEXT, Fecha DATE, Cantidad INTEGER, 
                      FOREIGN KEY(CodE) REFERENCES EMPLEADOS(CodE), 
                      FOREIGN KEY(CodB) REFERENCES BARES(CodB), 
                      FOREIGN KEY(CodC) REFERENCES CERVEZAS(CodC));
''')

# Inserción de datos
cervezas_data = [('01','Botella',0.2,3600), ('02','Botella',0.33,1200), ('03','Lata',0.33,2400), ('04','Botella',1,288), ('05','Barril',60,30)]
bares_data = [('001','11111111X','Stop','Villa Botijo'), ('002','22222222Y','Las Vegas','Villa Botijo'), ('003','-','Club Social','Las Ranas'), ('004','33333333Z','Otra Ronda','La Esponja')]
empleados_data = [(1,'Prudencio Caminero',120000), (2,'Vicente Merario',110000), (3,'Valentin Siempre',100000)]
reparto_data = [
    (1,'001','01','2005-10-21',240), (1,'001','02','2005-10-21',48), (1,'002','03','2005-10-22',60),
    (1,'004','05','2005-10-22',4), (2,'002','03','2005-10-22',48), (2,'002','05','2005-10-23',2),
    (2,'004','01','2005-10-23',480), (2,'004','02','2005-10-24',72), (3,'003','03','2005-10-24',48), (3,'003','04','2005-10-25',20)
]

cursor.executemany('INSERT INTO CERVEZAS VALUES (?,?,?,?)', cervezas_data)
cursor.executemany('INSERT INTO BARES VALUES (?,?,?,?)', bares_data)
cursor.executemany('INSERT INTO EMPLEADOS VALUES (?,?,?)', empleados_data)
cursor.executemany('INSERT INTO REPARTO VALUES (?,?,?,?,?)', reparto_data)
conexion.commit()

## Ejercicios
---
### 1. Nombre de los empleados que repartieron al bar Stop (17-23 oct 2005)

In [3]:
query = """
SELECT DISTINCT E.Nombre
FROM EMPLEADOS E
JOIN REPARTO R ON E.CodE = R.CodE
JOIN BARES B ON R.CodB = B.CodB
WHERE B.Nombre = 'Stop' AND R.Fecha BETWEEN '2005-10-17' AND '2005-10-23';
"""
pd.read_sql_query(query, conexion)

,Nombre
0,Prudencio Caminero


### 2. Cif y nombre de los bares con Botella < 1L, por localidad

In [4]:
query = """
SELECT DISTINCT B.Cif, B.Nombre
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella' AND C.Capacidad < 1
ORDER BY B.Localidad;
"""
pd.read_sql_query(query, conexion)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


### 3. Repartos de Prudencio Caminero

In [5]:
query = """
SELECT B.Nombre, C.Envase, C.Capacidad, R.Fecha, R.Cantidad
FROM REPARTO R
JOIN EMPLEADOS E ON R.CodE = E.CodE
JOIN BARES B ON R.CodB = B.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE E.Nombre = 'Prudencio Caminero';
"""
pd.read_sql_query(query, conexion)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4


### 4. Bares con envase botella y capacidad 0.2 o 0.33

In [8]:
query = """
SELECT DISTINCT B.Nombre
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella' AND C.Capacidad IN (0.2, 0.33);
"""
pd.read_sql_query(query, conexion)

,Nombre
0,Stop
1,Otra Ronda


### 5. Empleados que repartieron a 'Stop' y 'Las Vegas' (Botellas)

In [9]:
query = """
SELECT E.Nombre
FROM EMPLEADOS E
JOIN REPARTO R ON E.CodE = R.CodE
JOIN BARES B ON R.CodB = B.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella' AND B.Nombre IN ('Stop', 'Las Vegas')
GROUP BY E.Nombre
HAVING COUNT(DISTINCT B.Nombre) = 2;
"""
pd.read_sql_query(query, conexion)

,Nombre


### 6. Viajes fuera de Villa Botijo

In [7]:
query = """
SELECT E.Nombre, COUNT(R.CodE) AS NumeroViajes
FROM EMPLEADOS E
JOIN REPARTO R ON E.CodE = R.CodE
JOIN BARES B ON R.CodB = B.CodB
WHERE B.Localidad <> 'Villa Botijo'
GROUP BY E.Nombre;
"""
pd.read_sql_query(query, conexion)

,Nombre,NumeroViajes
0,Prudencio Caminero,1
1,Valentin Siempre,2
2,Vicente Merario,2


### 7. Bar que más litros ha comprado

In [10]:
query = """
SELECT B.Nombre, B.Localidad
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
GROUP BY B.CodB
ORDER BY SUM(R.Cantidad * C.Capacidad) DESC
LIMIT 1;
"""
pd.read_sql_query(query, conexion)

,Nombre,Localidad
0,Otra Ronda,La Esponja


### 8. Bares con todos los tipos de botella < 1L

In [11]:
query = """
SELECT B.Nombre
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella' AND C.Capacidad < 1
GROUP BY B.Nombre
HAVING COUNT(DISTINCT C.CodC) = (
    SELECT COUNT(*) FROM CERVEZAS WHERE Envase = 'Botella' AND Capacidad < 1
);
"""
pd.read_sql_query(query, conexion)

,Nombre
0,Otra Ronda
1,Stop


### 9. Subir sueldo al empleado que más días trabajó

In [12]:
# Primero identificamos al empleado y actualizamos
cursor.execute('''
UPDATE EMPLEADOS
SET Sueldo = Sueldo * 1.05
WHERE CodE = (
    SELECT CodE
    FROM REPARTO
    GROUP BY CodE
    ORDER BY COUNT(DISTINCT Fecha) DESC
    LIMIT 1
);
''')
conexion.commit()

# Mostramos el resultado
pd.read_sql_query("SELECT * FROM EMPLEADOS", conexion)

,CodE,Nombre,Sueldo
0,1,Prudencio Caminero,120000.0
1,2,Vicente Merario,115500.0
2,3,Valentin Siempre,100000.0


### 10. Insertar nuevo reparto

In [13]:
cursor.execute('''
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
VALUES (
    (SELECT CodE FROM EMPLEADOS WHERE Nombre = 'Vicente Merario'),
    (SELECT CodB FROM BARES WHERE Nombre = 'Stop'),
    (SELECT CodC FROM CERVEZAS WHERE Envase = 'Lata' LIMIT 1),
    '2005-10-26',
    48
);
''')
conexion.commit()

# Verificamos la inserción
pd.read_sql_query("SELECT * FROM REPARTO WHERE Fecha = '2005-10-26'", conexion)

,CodE,CodB,CodC,Fecha,Cantidad
0,2,001,03,2005-10-26,48
